# NBA 3.1 v5 — NBRx-Based Indication × Drug-Pool Analysis

**Key Changes from v4:**
1. **NBRx-Based Deciling** — `sum(NBRX)` from `lrx_nbrx` (same source, explicit NBRx metric)
2. **Kerendia Writer Coverage** — identify writers missed by deciling, develop rules to include them
3. **Non-Paid Claims** — expand Kerendia writer pool to include non-paid claims

**Source:** `pace_ker_in_house_schema.lrx_nbrx`  
**Indication column:** `INDICATION_GROUP_APP1` (T2D, CKD, HF, T1D — compound values with `contains()`)  
**Product column:** `PRODUCT_CLASS` (SGLT2, GLP1, ACE, ARB, KERENDIA, sMRA, etc.)  
**Drug pools:** SGLT2/GLP1, ACE/ARB — **sMRA excluded**

In [0]:
%run "/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
# ============================================================
# V5 Parameters
# ============================================================
import builtins
import itertools
from functools import reduce
from pyspark.sql import functions as F, Window

CATALOG = "ph-com-ai-us-prod-catalog-2553213839599380"
SCHEMA  = "pace_ker_in_house_schema"
SOURCE_TABLE = f"`{CATALOG}`.`{SCHEMA}`.lrx_nbrx"

LOOKBACK_START = "2024-10-01"
LOOKBACK_END   = "2026-09-30"

# --- PRODUCT_CLASS → Drug Pool Mapping ---
DRUG_POOLS = {
    "SGLT2_GLP1": ["SGLT2", "GLP1"],
    "ACE_ARB":    ["ACE", "ARB"],
}

INDICATIONS = ["T2D", "CKD", "HF", "T1D"]

# Column names
INDICATION_COL = "INDICATION_GROUP_APP1"
PRODUCT_COL    = "PRODUCT_CLASS"
METRIC_COL     = "NBRX"

print(f"V5 Parameters loaded.")
print(f"  Source: {SOURCE_TABLE}")
print(f"  Lookback: {LOOKBACK_START} – {LOOKBACK_END}")
print(f"  Indication column: {INDICATION_COL}")
print(f"  Product column: {PRODUCT_COL}")
print(f"  Metric: sum({METRIC_COL})")
print(f"  Indications: {INDICATIONS}")
print(f"  Drug pools: {list(DRUG_POOLS.keys())} (sMRA excluded)")

In [0]:
# ============================================================
# Load lrx_nbrx & apply lookback window
# ============================================================
df_src = spark.table(SOURCE_TABLE)
lrx = df_src.filter(
    (F.col("WEEK_ENDING_DATE") >= LOOKBACK_START) &
    (F.col("WEEK_ENDING_DATE") <= LOOKBACK_END)
)

print(f"Full table rows:     {df_src.count():,}")
print(f"Lookback rows:       {lrx.count():,}")
print(f"Unique HCPs:         {lrx.select('PROVIDER_ID').distinct().count():,}")

print(f"\nSchema:")
lrx.printSchema()

print(f"\n{INDICATION_COL} values:")
lrx.select(INDICATION_COL).distinct().orderBy(INDICATION_COL).show(20, truncate=False)

print(f"\n{PRODUCT_COL} values:")
lrx.select(PRODUCT_COL).distinct().orderBy(PRODUCT_COL).show(20, truncate=False)

In [0]:
# ============================================================
# PRODUCT_CLASS distribution (top by NBRX volume)
# ============================================================
pc_dist = (
    lrx.groupBy(PRODUCT_COL)
    .agg(
        F.sum(METRIC_COL).alias("TOTAL_NBRX"),
        F.countDistinct("PROVIDER_ID").alias("N_HCPS"),
    )
    .orderBy(F.desc("TOTAL_NBRX"))
)
display(pc_dist)

In [0]:
# ============================================================
# Cumulative volume-based decile function (NBRx)
# Each decile ≈ 10% of total NBRX volume
# Tiebreaker: PROVIDER_ID ascending
# ============================================================

def compute_deciles(df, indication, pool_products):
    """
    For a given indication × drug pool, compute cumulative
    volume-based deciles (1–10) per PROVIDER_ID.
    Returns a DataFrame with PROVIDER_ID, TOTAL_NBRX, DECILE.
    """
    filtered = df.filter(
        (F.col(INDICATION_COL).contains(indication)) &
        (F.col(PRODUCT_COL).isin(pool_products))
    )
    hcp_vol = filtered.groupBy("PROVIDER_ID").agg(
        F.sum(METRIC_COL).alias("TOTAL_NBRX")
    )
    total_vol = hcp_vol.agg(F.sum("TOTAL_NBRX")).collect()[0][0]
    if total_vol is None or total_vol == 0:
        return None, 0, 0
    vol_per_decile = float(total_vol) / 10.0

    w = Window.orderBy(F.asc("TOTAL_NBRX"), F.asc("PROVIDER_ID"))
    hcp_vol = hcp_vol.withColumn("CUM_NBRX", F.sum("TOTAL_NBRX").over(w))
    hcp_vol = hcp_vol.withColumn(
        "DECILE",
        F.least(
            (F.col("CUM_NBRX") / F.lit(vol_per_decile)).cast("int") + 1,
            F.lit(10)
        )
    ).drop("CUM_NBRX")

    total_hcps = hcp_vol.count()
    return hcp_vol, total_hcps, int(total_vol)

print("Decile function defined.")

In [0]:
# ============================================================
# Compute all 8 decile sets with full D1–D10 distribution
# ============================================================
all_decile_dfs = {}   # (indication, pool_name) -> decile DataFrame
all_stats = []        # list of summary dicts

for ind in INDICATIONS:
    for pool_name, pool_products in DRUG_POOLS.items():
        key = (ind, pool_name)
        dec_df, total_hcps, total_nbrx = compute_deciles(lrx, ind, pool_products)
        all_decile_dfs[key] = dec_df

        # Per-decile breakdown
        decile_rows = (
            dec_df.groupBy("DECILE")
            .agg(
                F.count("*").alias("N_HCPS"),
                F.sum("TOTAL_NBRX").alias("DECILE_NBRX"),
                F.min("TOTAL_NBRX").alias("MIN_NBRX"),
                F.max("TOTAL_NBRX").alias("MAX_NBRX"),
            )
            .orderBy("DECILE")
            .collect()
        )

        print(f"\n{'='*75}")
        print(f"{ind} × {pool_name}  |  Total HCPs: {total_hcps:,}  |  Total NBRx: {total_nbrx:,}")
        print(f"{'='*75}")
        print(f"{'Decile':>6} {'HCPs':>10} {'%HCPs':>7} {'NBRx':>12} {'Cum%HCPs':>10} {'Cum%NBRx':>10} {'Min':>8} {'Max':>8}")
        print("-" * 80)

        cum_hcps, cum_nbrx = 0, 0
        dec_summary = {}
        for d in decile_rows:
            cum_hcps += d["N_HCPS"]
            cum_nbrx += d["DECILE_NBRX"]
            dec_summary[d["DECILE"]] = d["N_HCPS"]
            print(
                f"{d['DECILE']:>6} {d['N_HCPS']:>10,} {100*d['N_HCPS']/total_hcps:>6.1f}% "
                f"{int(d['DECILE_NBRX']):>12,} "
                f"{100*cum_hcps/total_hcps:>9.1f}% {100*cum_nbrx/total_nbrx:>9.1f}% "
                f"{int(d['MIN_NBRX']):>8,} {int(d['MAX_NBRX']):>8,}"
            )

        all_stats.append({
            "indication": ind, "pool": pool_name,
            "total_hcps": total_hcps, "total_nbrx": total_nbrx,
            "dec_summary": dec_summary,
        })

In [0]:
# ============================================================
# Summary table: HCP counts by decile cutoff (D2+ through D5+)
# ============================================================
print(f"{'Ind × Pool':<25} {'Total':>10} {'D2+':>10} {'D3+':>10} {'D4+':>10} {'D5+':>10}")
print("=" * 70)

for s in all_stats:
    total = s["total_hcps"]
    d1 = s["dec_summary"].get(1, 0)
    d2 = s["dec_summary"].get(2, 0)
    d3 = s["dec_summary"].get(3, 0)
    d2plus = total - d1
    d3plus = d2plus - d2
    d4plus = d3plus - d3
    d5plus = d4plus - s["dec_summary"].get(4, 0)
    print(
        f"{s['indication']} × {s['pool']:<16} {total:>10,} "
        f"{d2plus:>10,} {d3plus:>10,} {d4plus:>10,} {d5plus:>10,}"
    )

## Kerendia Writer Coverage Analysis

Identify Kerendia writers from `lrx_nbrx`:  
* **Paid writers:** `PRODUCT_CLASS = 'KERENDIA'` AND `NBRX > 0`  
* **All writers:** `PRODUCT_CLASS = 'KERENDIA'` (includes zero-NBRX rows if any)

In [0]:
# ============================================================
# Kerendia Writers: Paid vs All (incl non-paid)
# ============================================================

# Paid writers (NBRX > 0)
ker_paid = (
    lrx.filter((F.col(PRODUCT_COL) == "KERENDIA") & (F.col(METRIC_COL) > 0))
    .select("PROVIDER_ID").distinct()
)
ker_paid_ct = ker_paid.count()

# All Kerendia claims (incl non-paid)
ker_all = lrx.filter(F.col(PRODUCT_COL) == "KERENDIA").select("PROVIDER_ID").distinct()
ker_all_ct = ker_all.count()

print(f"Kerendia writers (paid NBRX > 0):        {ker_paid_ct:,}")
print(f"Kerendia writers (all claims, incl non-paid): {ker_all_ct:,}")
print(f"Incremental from non-paid:                  +{ker_all_ct - ker_paid_ct:,} ({100*(ker_all_ct - ker_paid_ct)/ker_paid_ct:.1f}%)")

print(f"\nKerendia claims by PAY_TYPE:")
ker_pay = (
    lrx.filter(F.col(PRODUCT_COL) == "KERENDIA")
    .groupBy(INDICATION_COL)
    .agg(
        F.sum(METRIC_COL).alias("NBRX"),
        F.countDistinct("PROVIDER_ID").alias("N_HCPS"),
    )
    .orderBy(F.desc("NBRX"))
)
print(f"\nKerendia writers by {INDICATION_COL}:")
display(ker_pay)

In [0]:
# ============================================================
# Kerendia Writer Capture Rate by Decile Cutoff (D2+ – D5+)
# For each of the 8 decile sets, check how many of the paid
# Kerendia writers fall in each decile tier.
# ============================================================
print(f"Total paid Kerendia writers: {ker_paid_ct:,}")
print(f"\n{'Ind × Pool':<25} {'In Set':>8} {'NOT in':>7}  {'D2+':>12} {'D3+':>12} {'D4+':>12} {'D5+':>12}")
print("=" * 95)

for ind in INDICATIONS:
    for pool_name in DRUG_POOLS:
        key = (ind, pool_name)
        dec_df = all_decile_dfs[key]

        # Join Kerendia writers with their decile in this set
        ker_dec = dec_df.join(ker_paid, on="PROVIDER_ID", how="inner")
        in_set = ker_dec.count()
        not_in = ker_paid_ct - in_set

        # Count writers at each cutoff
        dec_dist = ker_dec.groupBy("DECILE").count().orderBy("DECILE").collect()
        dec_dict = {r["DECILE"]: r["count"] for r in dec_dist}

        results = {}
        for cutoff in [2, 3, 4, 5]:
            captured = builtins.sum(v for k, v in dec_dict.items() if k >= cutoff)
            pct = 100 * captured / ker_paid_ct
            results[cutoff] = (captured, pct)

        print(
            f"{ind} × {pool_name:<16} {in_set:>8,} {not_in:>7,}  "
            f"{results[2][0]:>7,} ({results[2][1]:.1f}%) "
            f"{results[3][0]:>7,} ({results[3][1]:.1f}%) "
            f"{results[4][0]:>7,} ({results[4][1]:.1f}%) "
            f"{results[5][0]:>7,} ({results[5][1]:.1f}%)"
        )

In [0]:
# ============================================================
# Profile Kerendia writers NOT captured by ANY decile set
# These are writers with zero claims in all 8 ind×pool combos
# ============================================================

# Union of all 8 decile sets
all_deciled_hcps = reduce(
    lambda a, b: a.unionByName(b),
    [all_decile_dfs[k].select("PROVIDER_ID") for k in all_decile_dfs]
).distinct()

ker_not_in_any = ker_paid.join(all_deciled_hcps, on="PROVIDER_ID", how="left_anti")
ker_not_ct = ker_not_in_any.count()
print(f"Kerendia writers NOT in ANY of 8 decile sets: {ker_not_ct:,}")

if ker_not_ct > 0:
    # Check their specialty
    specialty = lrx.select("PROVIDER_ID", "SPECIALTY_GROUP").distinct()
    missed_spec = (
        ker_not_in_any.join(specialty, on="PROVIDER_ID", how="left")
        .groupBy("SPECIALTY_GROUP")
        .agg(F.countDistinct("PROVIDER_ID").alias("N_HCPS"))
        .orderBy(F.desc("N_HCPS"))
    )
    print(f"\nMissed Kerendia writers by specialty:")
    display(missed_spec)

    # Check their Kerendia NBRX volume
    missed_vol = (
        lrx.filter(F.col(PRODUCT_COL) == "KERENDIA")
        .join(ker_not_in_any, on="PROVIDER_ID", how="inner")
        .groupBy("PROVIDER_ID")
        .agg(F.sum(METRIC_COL).alias("KERENDIA_NBRX"))
    )
    print(f"\nMissed writers Kerendia NBRX distribution:")
    missed_vol.select(
        F.count("*").alias("N"),
        F.mean("KERENDIA_NBRX").alias("AVG"),
        F.expr("percentile_approx(KERENDIA_NBRX, 0.5)").alias("MEDIAN"),
        F.min("KERENDIA_NBRX").alias("MIN"),
        F.max("KERENDIA_NBRX").alias("MAX"),
    ).show()

In [0]:
# ============================================================
# Best possible Kerendia capture: union across all 8 sets
# at each decile cutoff
# ============================================================
print(f"\nKerendia writer capture — UNION of all 8 sets:")
print(f"{'Cutoff':<10} {'Captured':>10} {'% of Paid Writers':>18}")
print("-" * 40)

for cutoff in [2, 3, 4, 5]:
    union_captured = reduce(
        lambda a, b: a.unionByName(b),
        [
            all_decile_dfs[k].filter(F.col("DECILE") >= cutoff)
            .join(ker_paid, on="PROVIDER_ID", how="inner")
            .select("PROVIDER_ID")
            for k in all_decile_dfs
        ]
    ).distinct()
    ct = union_captured.count()
    print(f"D{cutoff}+       {ct:>10,} {100*ct/ker_paid_ct:>17.1f}%")

## Kerendia Writer Profiling by Set

**Core sets (from NBA 3.0):**
* **Set 1:** T2D × SGLT2/GLP1 (present in NBA 3.0 via TRx)
* **Set 3:** CKD × SGLT2/GLP1 (present in NBA 3.0 via TRx)

**Goal:** Profile missing Kerendia writers in each set to find patterns that inform:
1. The right decile cutoff for each set
2. The right intersection/union strategy with the core sets

In [0]:
# ============================================================
# For each set: full decile distribution of Kerendia writers
# Shows where writers land in each set (D1 = low volume, D10 = high)
# ============================================================
SET_LABELS = {
    ("T2D", "SGLT2_GLP1"): "Set1: T2D×SGLT2/GLP1 (CORE)",
    ("T2D", "ACE_ARB"):    "Set2: T2D×ACE/ARB",
    ("CKD", "SGLT2_GLP1"): "Set3: CKD×SGLT2/GLP1 (CORE)",
    ("CKD", "ACE_ARB"):    "Set4: CKD×ACE/ARB",
    ("HF",  "SGLT2_GLP1"): "Set5: HF×SGLT2/GLP1",
    ("HF",  "ACE_ARB"):    "Set6: HF×ACE/ARB",
    ("T1D", "SGLT2_GLP1"): "Set7: T1D×SGLT2/GLP1",
    ("T1D", "ACE_ARB"):    "Set8: T1D×ACE/ARB",
}

for key, label in SET_LABELS.items():
    dec_df = all_decile_dfs[key]
    total_in_set = dec_df.count()
    
    # Join with Kerendia writers
    ker_in = dec_df.join(ker_paid, on="PROVIDER_ID", how="inner")
    ker_in_ct = ker_in.count()
    ker_missing = ker_paid_ct - ker_in_ct
    
    # Decile distribution of writers IN this set
    dec_dist = (
        ker_in.groupBy("DECILE")
        .agg(F.count("*").alias("KER_WRITERS"))
        .orderBy("DECILE")
        .collect()
    )
    
    print(f"\n{'='*75}")
    print(f"{label}")
    print(f"Total HCPs in set: {total_in_set:,}  |  Kerendia writers in set: {ker_in_ct:,}  |  Missing: {ker_missing:,}")
    print(f"{'='*75}")
    print(f"{'Decile':>6} {'KerWriters':>12} {'Cum Writers':>12} {'Cum % of All':>13} {'Set HCPs at D':>14} {'Writer Density':>15}")
    print("-" * 75)
    
    cum_ker = 0
    # Get total HCPs per decile for density calc
    set_dec = {r["DECILE"]: r["N_HCPS"] for r in dec_df.groupBy("DECILE").agg(F.count("*").alias("N_HCPS")).collect()}
    
    for d in dec_dist:
        dec = d["DECILE"]
        kw = d["KER_WRITERS"]
        cum_ker += kw
        set_hcps = set_dec.get(dec, 0)
        density = 100 * kw / set_hcps if set_hcps > 0 else 0
        print(
            f"{dec:>6} {kw:>12,} {cum_ker:>12,} "
            f"{100*cum_ker/ker_paid_ct:>12.1f}% "
            f"{set_hcps:>14,} "
            f"{density:>14.2f}%"
        )

In [0]:
# ============================================================
# For writers MISSING from each set:
# - Which other sets contain them?
# - At what decile?
# - What's their specialty?
# ============================================================
specialty = lrx.select("PROVIDER_ID", "SPECIALTY_GROUP").distinct()

for key, label in SET_LABELS.items():
    dec_df = all_decile_dfs[key]
    
    # Writers missing from this set
    ker_missing = ker_paid.join(dec_df.select("PROVIDER_ID"), on="PROVIDER_ID", how="left_anti")
    miss_ct = ker_missing.count()
    
    if miss_ct == 0:
        print(f"\n{label}: 0 missing writers")
        continue
    
    print(f"\n{'='*80}")
    print(f"{label} — {miss_ct:,} MISSING Kerendia writers")
    print(f"{'='*80}")
    
    # Specialty of missing writers
    miss_spec = (
        ker_missing.join(specialty, on="PROVIDER_ID", how="left")
        .groupBy("SPECIALTY_GROUP")
        .agg(F.countDistinct("PROVIDER_ID").alias("N"))
        .orderBy(F.desc("N"))
        .collect()
    )
    print(f"  Specialty: ", end="")
    print(", ".join(f"{r['SPECIALTY_GROUP']}:{r['N']}" for r in miss_spec[:6]))
    
    # Kerendia NBRX of missing writers
    miss_vol = (
        lrx.filter(F.col(PRODUCT_COL) == "KERENDIA")
        .join(ker_missing, on="PROVIDER_ID", how="inner")
        .groupBy("PROVIDER_ID")
        .agg(F.sum(METRIC_COL).alias("K_NBRX"))
        .agg(
            F.mean("K_NBRX").alias("AVG"),
            F.expr("percentile_approx(K_NBRX, 0.5)").alias("MEDIAN"),
            F.max("K_NBRX").alias("MAX"),
        )
        .collect()[0]
    )
    print(f"  Kerendia NBRX: Avg={float(miss_vol['AVG']):.1f}, Median={int(miss_vol['MEDIAN'])}, Max={int(miss_vol['MAX'])}")
    
    # Which OTHER sets capture these missing writers, and at what decile?
    print(f"  Cross-set recovery:")
    for other_key, other_label in SET_LABELS.items():
        if other_key == key:
            continue
        other_df = all_decile_dfs[other_key]
        recovered = other_df.join(ker_missing, on="PROVIDER_ID", how="inner")
        rec_ct = recovered.count()
        if rec_ct == 0:
            continue
        # At what deciles?
        rec_dec = recovered.groupBy("DECILE").count().orderBy("DECILE").collect()
        d2plus = builtins.sum(r["count"] for r in rec_dec if r["DECILE"] >= 2)
        d3plus = builtins.sum(r["count"] for r in rec_dec if r["DECILE"] >= 3)
        print(
            f"    {other_label:<35} recovers {rec_ct:>5,}/{miss_ct}  "
            f"(D2+: {d2plus:>5,}, D3+: {d3plus:>5,})"
        )

In [0]:
# ============================================================
# Core = Set1 (T2D×SGLT2/GLP1) ∪ Set3 (CKD×SGLT2/GLP1)
# at each decile cutoff — how many writers captured?
# Then: what's the marginal gain of adding each other set?
# ============================================================
print("CORE SETS: Set1 (T2D×SGLT2/GLP1) ∪ Set3 (CKD×SGLT2/GLP1)")
print("=" * 80)

set1 = all_decile_dfs[("T2D", "SGLT2_GLP1")]
set3 = all_decile_dfs[("CKD", "SGLT2_GLP1")]

for cutoff in [2, 3, 4, 5]:
    core_hcps = (
        set1.filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID")
        .unionByName(set3.filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID"))
        .distinct()
    )
    core_ct = core_hcps.count()
    core_ker = core_hcps.join(ker_paid, on="PROVIDER_ID", how="inner").count()
    core_gap = ker_paid_ct - core_ker
    
    print(f"\nD{cutoff}+ Core: {core_ct:,} HCPs | Kerendia: {core_ker:,} ({100*core_ker/ker_paid_ct:.1f}%) | Gap: {core_gap:,}")
    
    # For each other set, what's the marginal gain?
    gap_writers = ker_paid.join(core_hcps, on="PROVIDER_ID", how="left_anti")
    
    marginals = []
    for other_key, other_label in SET_LABELS.items():
        if other_key in [("T2D", "SGLT2_GLP1"), ("CKD", "SGLT2_GLP1")]:
            continue
        other_df = all_decile_dfs[other_key]
        # Try adding this set at same cutoff
        other_hcps = other_df.filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID")
        other_ct = other_hcps.count()
        # How many gap writers does it recover?
        recovered = gap_writers.join(other_hcps, on="PROVIDER_ID", how="inner").count()
        marginals.append((other_label, other_ct, recovered))
    
    marginals.sort(key=lambda x: -x[2])  # sort by recovery
    print(f"  {'Add-on Set':<35} {'Set HCPs':>10} {'Gap Recovered':>14} {'Recovery %':>11}")
    print(f"  {'-'*72}")
    for lbl, s_ct, rec in marginals:
        print(f"  {lbl:<35} {s_ct:>10,} {rec:>14,} {100*rec/core_gap:>10.1f}%")

In [0]:
# ============================================================
# Greedy strategy: start with Core (Set1 ∪ Set3), then add
# sets one at a time in order of best writer-recovery-per-HCP
# ratio. Shows the Pareto frontier.
# ============================================================
print("GREEDY SET ADDITION (starting from Core = Set1 ∪ Set3)")
print("=" * 90)

for cutoff in [2, 3, 4]:
    print(f"\n{'~'*90}")
    print(f"DECILE CUTOFF: D{cutoff}+")
    print(f"{'~'*90}")
    
    # Start with core
    current_hcps = (
        set1.filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID")
        .unionByName(set3.filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID"))
        .distinct()
    )
    current_ct = current_hcps.count()
    current_ker = current_hcps.join(ker_paid, on="PROVIDER_ID", how="inner").count()
    
    print(f"{'Step':<6} {'Action':<35} {'Universe':>10} {'Ker Writers':>12} {'% Captured':>11} {'Marginal HCPs':>14} {'Marginal Ker':>13}")
    print("-" * 105)
    print(f"{'0':<6} {'Core (Set1 ∪ Set3)':<35} {current_ct:>10,} {current_ker:>12,} {100*current_ker/ker_paid_ct:>10.1f}% {'-':>14} {'-':>13}")
    
    remaining_sets = [
        k for k in SET_LABELS.keys()
        if k not in [("T2D", "SGLT2_GLP1"), ("CKD", "SGLT2_GLP1")]
    ]
    added = []
    
    for step in range(1, len(remaining_sets) + 1):
        best_key, best_label, best_ratio = None, None, -1
        best_new_hcps, best_new_ker = None, None
        best_marginal_hcps, best_marginal_ker = 0, 0
        
        for k in remaining_sets:
            if k in added:
                continue
            add_hcps = all_decile_dfs[k].filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID")
            combined = current_hcps.unionByName(add_hcps).distinct()
            new_ct = combined.count()
            new_ker = combined.join(ker_paid, on="PROVIDER_ID", how="inner").count()
            m_hcps = new_ct - current_ct
            m_ker = new_ker - current_ker
            ratio = m_ker / m_hcps if m_hcps > 0 else 0
            if ratio > best_ratio:
                best_key, best_label = k, SET_LABELS[k]
                best_ratio = ratio
                best_new_hcps, best_new_ker = combined, new_ct
                best_marginal_hcps, best_marginal_ker = m_hcps, m_ker
                best_new_ker_writers = new_ker
        
        if best_key is None:
            break
        
        added.append(best_key)
        current_hcps = best_new_hcps
        current_ct = best_new_ker
        current_ker = best_new_ker_writers
        
        print(
            f"{step:<6} {'+ ' + best_label:<35} {current_ct:>10,} {current_ker:>12,} "
            f"{100*current_ker/ker_paid_ct:>10.1f}% {best_marginal_hcps:>14,} {best_marginal_ker:>13,}"
        )

## NBA 3.0 Universe Overlap Analysis

Compare v5 decile sets against the NBA 3.0 baseline universe (232,965 HCPs).  
**Source:** `kerendia_nba_3_hcp_model_base_hcp_universe_for_inference` (PULL_DT: 2026_10_09)  
**Flags:** KERENDIA_FLAG, SGLT2_GLP1_FLAG, NEPH_FLAG, TARGETING_HCP_FLAG, BPT_HCP_FLAG

In [0]:
# ============================================================
# Load NBA 3.0 Universe
# ============================================================
NBA30_PATH = (
    "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/"
    "dev_folder_3_1/notebook_output/"
    "kerendia_nba_3_hcp_model_base_hcp_universe_for_inference"
)
nba30 = spark.read.format("delta").load(NBA30_PATH)
nba30_ct = nba30.count()

print(f"NBA 3.0 Universe: {nba30_ct:,} HCPs")
print(f"\nFlag counts:")
for flag in ["KERENDIA_FLAG", "SGLT2_GLP1_FLAG", "NEPH_FLAG", "TARGETING_HCP_FLAG", "BPT_HCP_FLAG"]:
    ct = nba30.filter(F.col(flag) == 1).count()
    print(f"  {flag} = 1: {ct:,} ({100*ct/nba30_ct:.1f}%)")

# NBA 3.0 Kerendia writers
nba30_ker = nba30.filter(F.col("KERENDIA_FLAG") == 1).select("PROVIDER_ID")
nba30_ker_ct = nba30_ker.count()
print(f"\nNBA 3.0 Kerendia writers: {nba30_ker_ct:,}")
print(f"v5 Kerendia writers (paid NBRX>0): {ker_paid_ct:,}")

In [0]:
# ============================================================
# For each decile cutoff, compute:
# 1. v5 set overlap with NBA 3.0 (total HCPs)
# 2. Kerendia writer overlap
# 3. Flag distribution of overlapping HCPs
# ============================================================
nba30_ids = nba30.select("PROVIDER_ID")

print("v5 CORE (Set1 ∪ Set3) vs NBA 3.0 Universe")
print("=" * 95)
print(f"{'Cutoff':<8} {'v5 HCPs':>10} {'Overlap':>10} {'v5-only':>10} {'NBA30-only':>10} "
      f"{'Ker Ovlp':>10} {'Ker v5only':>10} {'Ker NBA30only':>14}")
print("-" * 95)

for cutoff in [1, 2, 3, 4, 5]:
    v5_core = (
        set1.filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID")
        .unionByName(set3.filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID"))
        .distinct()
    )
    v5_ct = v5_core.count()
    
    overlap = v5_core.join(nba30_ids, on="PROVIDER_ID", how="inner")
    overlap_ct = overlap.count()
    v5_only = v5_ct - overlap_ct
    nba30_only = nba30_ct - overlap_ct
    
    # Kerendia writer overlap
    v5_ker = v5_core.join(ker_paid, on="PROVIDER_ID", how="inner")
    v5_ker_ct = v5_ker.count()
    
    ker_overlap = v5_ker.join(nba30_ker, on="PROVIDER_ID", how="inner").count()
    ker_v5_only = v5_ker_ct - ker_overlap
    ker_nba30_only = nba30_ker_ct - ker_overlap
    
    label = f"D{cutoff}+" if cutoff > 1 else "ALL"
    print(
        f"{label:<8} {v5_ct:>10,} {overlap_ct:>10,} {v5_only:>10,} {nba30_only:>10,} "
        f"{ker_overlap:>10,} {ker_v5_only:>10,} {ker_nba30_only:>14,}"
    )

In [0]:
# ============================================================
# For the Core (Set1 ∪ Set3) at each cutoff:
# Show flag counts of the OVERLAPPING HCPs with NBA 3.0
# ============================================================
FLAGS = ["KERENDIA_FLAG", "SGLT2_GLP1_FLAG", "NEPH_FLAG", "TARGETING_HCP_FLAG", "BPT_HCP_FLAG"]

for cutoff in [2, 3, 4]:
    v5_core = (
        set1.filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID")
        .unionByName(set3.filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID"))
        .distinct()
    )
    
    overlap = v5_core.join(nba30, on="PROVIDER_ID", how="inner")
    overlap_ct = overlap.count()
    v5_only = v5_core.join(nba30_ids, on="PROVIDER_ID", how="left_anti")
    v5_only_ct = v5_only.count()
    nba30_only = nba30.join(v5_core, on="PROVIDER_ID", how="left_anti")
    nba30_only_ct = nba30_only.count()
    
    print(f"\n{'='*80}")
    print(f"CORE D{cutoff}+ vs NBA 3.0  |  Overlap: {overlap_ct:,}  |  v5-only: {v5_only_ct:,}  |  NBA30-only: {nba30_only_ct:,}")
    print(f"{'='*80}")
    
    # Flag counts in overlap
    print(f"\n  {'Flag':<22} {'In Overlap':>10} {'% of Ovlp':>10} {'In NBA30-only':>14} {'% of NBA30only':>15}")
    print(f"  {'-'*73}")
    for flag in FLAGS:
        ovlp_flag = overlap.filter(F.col(flag) == 1).count()
        nba30_flag = nba30_only.filter(F.col(flag) == 1).count()
        print(
            f"  {flag:<22} {ovlp_flag:>10,} {100*ovlp_flag/overlap_ct:>9.1f}% "
            f"{nba30_flag:>14,} {100*nba30_flag/nba30_only_ct:>14.1f}%"
        )
    
    # Kerendia writer detail
    ker_in_overlap = overlap.filter(F.col("KERENDIA_FLAG") == 1).count()
    ker_in_v5only = v5_only.join(ker_paid, on="PROVIDER_ID", how="inner").count()
    ker_in_nba30only = nba30_only.filter(F.col("KERENDIA_FLAG") == 1).count()
    
    print(f"\n  Kerendia writer breakdown:")
    print(f"    In overlap:   {ker_in_overlap:,}")
    print(f"    v5-only:      {ker_in_v5only:,} (new writers not in NBA 3.0)")
    print(f"    NBA30-only:   {ker_in_nba30only:,} (writers LOST if switching to v5)")

In [0]:
# ============================================================
# Full 8-set union at each cutoff vs NBA 3.0
# ============================================================
print("FULL 8-SET UNION vs NBA 3.0")
print("=" * 95)
print(f"{'Cutoff':<8} {'v5 Union':>10} {'Overlap':>10} {'%NBA30':>8} {'v5-only':>10} {'NBA30-only':>10} "
      f"{'KerOvlp':>8} {'KerV5only':>10} {'KerNBA30only':>13}")
print("-" * 95)

for cutoff in [2, 3, 4]:
    v5_union = reduce(
        lambda a, b: a.unionByName(b),
        [all_decile_dfs[k].filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID")
         for k in all_decile_dfs]
    ).distinct()
    v5_ct = v5_union.count()
    
    overlap_ct = v5_union.join(nba30_ids, on="PROVIDER_ID", how="inner").count()
    v5_only = v5_ct - overlap_ct
    nba30_only = nba30_ct - overlap_ct
    
    # Kerendia
    v5_ker_ct = v5_union.join(ker_paid, on="PROVIDER_ID", how="inner").count()
    ker_ovlp = v5_union.join(ker_paid, on="PROVIDER_ID", how="inner") \
                       .join(nba30_ker, on="PROVIDER_ID", how="inner").count()
    ker_v5only = v5_ker_ct - ker_ovlp
    ker_nba30only = nba30_ker_ct - ker_ovlp
    
    print(
        f"D{cutoff}+     {v5_ct:>10,} {overlap_ct:>10,} {100*overlap_ct/nba30_ct:>7.1f}% {v5_only:>10,} {nba30_only:>10,} "
        f"{ker_ovlp:>8,} {ker_v5only:>10,} {ker_nba30only:>13,}"
    )

## Combined Universe: Core + Criterion 2 + Criterion 3
* **C1 (Core):** Set1 (T2D×SGLT2/GLP1) ∪ Set3 (CKD×SGLT2/GLP1) at decile cutoff
* **C2:** Kerendia Writers (NBRX>0) = 40,112
* **C3:** NEPH (15,048) ∪ CARD (51,666) = 66,714

In [0]:
# ============================================================
# Define Criterion 2 & 3, build combined, compute NBA 3.0 overlap
# ============================================================
ker_c2 = lrx.filter((F.col(PRODUCT_COL) == "KERENDIA") & (F.col(METRIC_COL) > 0)).select("PROVIDER_ID").distinct()
neph_c3 = lrx.filter(F.col("SPECIALTY_GROUP") == "NEPH").select("PROVIDER_ID").distinct()
card_c3 = lrx.filter(F.col("SPECIALTY_GROUP") == "CARD").select("PROVIDER_ID").distinct()

print(f"C2 Kerendia Writers: {ker_c2.count():,}")
print(f"C3 NEPH: {neph_c3.count():,}, CARD: {card_c3.count():,}, Union: {neph_c3.unionByName(card_c3).distinct().count():,}")
print(f"Kerendia∩NEPH: {ker_c2.join(neph_c3, on='PROVIDER_ID', how='inner').count():,}")
print(f"Kerendia∩CARD: {ker_c2.join(card_c3, on='PROVIDER_ID', how='inner').count():,}")

nba30_ids_c = nba30.select("PROVIDER_ID")
print(f"\n{'='*110}")
print(f"COMBINED: Core (Set1 ∪ Set3) + C2 (Kerendia) + C3 (NEPH ∪ CARD)")
print(f"{'='*110}")
print(f"{'Cutoff':<8} {'C1':>10} {'+C2':>10} {'+NEPH':>10} {'+CARD=Total':>12} {'NBA30 Ovlp':>11} {'%NBA30':>7} {'v5-only':>9} {'NBA30-only':>11}")
print("-" * 100)

for cutoff in [2, 3, 4, 5]:
    c1 = set1.filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID") \
        .unionByName(set3.filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID")).distinct()
    c12 = c1.unionByName(ker_c2).distinct()
    c12n = c12.unionByName(neph_c3).distinct()
    full = c12n.unionByName(card_c3).distinct()
    fc = full.count()
    ov = full.join(nba30_ids_c, on="PROVIDER_ID", how="inner").count()
    print(f"D{cutoff}+     {c1.count():>10,} {c12.count():>10,} {c12n.count():>10,} {fc:>12,} {ov:>11,} {100*ov/nba30_ct:>6.1f}% {fc-ov:>9,} {nba30_ct-ov:>11,}")

In [0]:
# ============================================================
# Flag distribution + Kerendia writer coverage per cutoff
# ============================================================
FLAGS_C = ["KERENDIA_FLAG", "SGLT2_GLP1_FLAG", "NEPH_FLAG", "TARGETING_HCP_FLAG", "BPT_HCP_FLAG"]

for cutoff in [2, 3, 4]:
    c1 = set1.filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID") \
        .unionByName(set3.filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID")).distinct()
    full = c1.unionByName(ker_c2).unionByName(neph_c3).unionByName(card_c3).distinct()
    fc = full.count()
    ovlp = full.join(nba30, on="PROVIDER_ID", how="inner")
    oc = ovlp.count()
    n30o = nba30.join(full, on="PROVIDER_ID", how="left_anti")
    nc = n30o.count()
    v5o = full.join(nba30_ids_c, on="PROVIDER_ID", how="left_anti")

    print(f"\n{'='*85}")
    print(f"COMBINED D{cutoff}+ ({fc:,}) | Overlap: {oc:,} ({100*oc/nba30_ct:.1f}%) | v5-only: {v5o.count():,} | NBA30-only: {nc:,}")
    print(f"{'='*85}")
    print(f"  {'Flag':<22} {'Overlap':>10} {'%':>7} {'NBA30only':>10} {'%':>7}")
    print(f"  {'-'*58}")
    for f_name in FLAGS_C:
        of = ovlp.filter(F.col(f_name)==1).count()
        nf = n30o.filter(F.col(f_name)==1).count()
        print(f"  {f_name:<22} {of:>10,} {100*of/oc:>6.1f}% {nf:>10,} {100*nf/nc:>6.1f}%")
    kf = full.join(ker_c2, on="PROVIDER_ID", how="inner").count()
    kn = n30o.filter(F.col('KERENDIA_FLAG')==1).count()
    print(f"\n  Kerendia: {kf:,}/{ker_paid_ct:,} ({100*kf/ker_paid_ct:.1f}%) in universe | NBA30-only KER: {kn:,}")

## 4 Kerendia Writer Metrics
* **M1:** NBA 3.0 KERENDIA_FLAG = 77,972
* **M2:** LAAD V6 NBRX>0, 24-mo lookback = 40,122
* **M3:** LAAD V6 TRX>0, 24-mo lookback = 67,835
* **M4:** Raw FCT_RX paid TRX (Snowflake), 24-mo lookback = 67,796

In [0]:
# ============================================================
# 3 Kerendia writer metrics for the combined universe
# M1: NBA 3.0 KERENDIA_FLAG (77,972)
# M2: LAAD V6 NBRX>0, 24-month lookback (40,122)
# M3: LAAD V6 TRX>0, 24-month lookback
# ============================================================
LAAD_V6_TBL = f"`{CATALOG}`.kerendia_laad_based_lrx.kerendia_laad_based_lrx_v6"
laad_v6 = spark.table(LAAD_V6_TBL)

nba30_ker_m1 = nba30.filter(F.col("KERENDIA_FLAG") == 1).select("PROVIDER_ID")
m1_ct = nba30_ker_m1.count()

m2_writers = laad_v6.filter(
    (F.col("WEEK_ENDING_DATE") >= "20241001") &
    (F.col("WEEK_ENDING_DATE") <= "20260930") &
    (F.col("PRODUCT_GROUP") == "KERENDIA") & (F.col("NBRX") > 0)
).select("PROVIDER_ID").distinct()
m2_ct = m2_writers.count()

m3_writers = laad_v6.filter(
    (F.col("WEEK_ENDING_DATE") >= "20241001") &
    (F.col("WEEK_ENDING_DATE") <= "20260930") &
    (F.col("PRODUCT_GROUP") == "KERENDIA") & (F.col("TRX") > 0)
).select("PROVIDER_ID").distinct()
m3_ct = m3_writers.count()

print(f"Kerendia Baselines: M1={m1_ct:,} | M2={m2_ct:,} | M3={m3_ct:,}")

print(f"\n{'='*120}")
print(f"COMBINED (C1+C2+C3): 3 Kerendia Capture Metrics")
print(f"{'='*120}")
print(f"{'Cut':<5} {'Total':>9} {'NBA30ovlp':>10} {'%NBA30':>7} | "
      f"{'M1:FLAG':>8} {'%M1':>6} | {'M2:V6nbrx':>10} {'%M2':>6} | {'M3:V6trx':>9} {'%M3':>6}")
print("-" * 120)

for cutoff in [2, 3, 4, 5]:
    c1 = set1.filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID") \
        .unionByName(set3.filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID")).distinct()
    full = c1.unionByName(ker_c2).unionByName(neph_c3).unionByName(card_c3).distinct()
    fc = full.count()
    ov = full.join(nba30_ids_c, on="PROVIDER_ID", how="inner").count()
    cap1 = full.join(nba30_ker_m1, on="PROVIDER_ID", how="inner").count()
    cap2 = full.join(m2_writers, on="PROVIDER_ID", how="inner").count()
    cap3 = full.join(m3_writers, on="PROVIDER_ID", how="inner").count()
    print(f"D{cutoff}+  {fc:>9,} {ov:>10,} {100*ov/nba30_ct:>6.1f}% | "
          f"{cap1:>8,} {100*cap1/m1_ct:>5.1f}% | {cap2:>10,} {100*cap2/m2_ct:>5.1f}% | "
          f"{cap3:>9,} {100*cap3/m3_ct:>5.1f}%")

In [0]:
# ============================================================
# M4: Kerendia writers from raw IQVIA LAAD FCT_RX (Snowflake)
# Replicates NBA 3.0 logic: FCT_RX JOIN DIM_PRD on NDC_CD
# CLAIM_TYPE='PD', PRODUCT_GROUP='KERENDIA', full history
# ============================================================
query_m4 = f'''
    SELECT DISTINCT A.PROVIDER_ID
    FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX A
    INNER JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD B
    USING (NDC_CD)
    WHERE A.CLAIM_TYPE = 'PD'
    AND A.SVC_DT >= '{LOOKBACK_START}'
    AND A.SVC_DT <= '{LOOKBACK_END}'
    AND UPPER(TRIM(B.PRODUCT_GROUP)) = 'KERENDIA'
'''

m4_raw = Get_Data_Snowflakes(query_m4)
m4_writers = m4_raw.select(F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID")).distinct()
m4_ct = m4_writers.count()

print(f"M4: FCT_RX Kerendia paid TRX (24-mo lookback): {m4_ct:,}")
print(f"\nAll Kerendia baselines:")
print(f"  M1: NBA 3.0 KERENDIA_FLAG:         {m1_ct:,}")
print(f"  M2: LAAD V6 NBRX>0 (24-mo):        {m2_ct:,}")
print(f"  M3: LAAD V6 TRX>0 (full history):   {m3_ct:,}")
print(f"  M4: FCT_RX paid TRX (24-mo lookback): {m4_ct:,}")

In [0]:
# ============================================================
# 4 Kerendia writer metrics for the combined universe
# M1: NBA 3.0 KERENDIA_FLAG (77,972)
# M2: LAAD V6 NBRX>0, 24-month lookback (40,122)
# M3: LAAD V6 TRX>0, full history (77,582)
# M4: FCT_RX paid TRX, full history (from Snowflake)
# ============================================================
print(f"Baselines: M1={m1_ct:,} | M2={m2_ct:,} | M3={m3_ct:,} | M4={m4_ct:,}")

print(f"\n{'='*140}")
print(f"COMBINED (C1+C2+C3): 4 Kerendia Capture Metrics")
print(f"{'='*140}")
print(f"{'Cut':<5} {'Total':>9} {'NBA30ovlp':>10} {'%NBA30':>7} | "
      f"{'M1:FLAG':>8} {'%M1':>6} | {'M2:V6nbrx':>10} {'%M2':>6} | "
      f"{'M3:V6trx':>9} {'%M3':>6} | {'M4:FCTrx':>9} {'%M4':>6}")
print("-" * 140)

for cutoff in [2, 3, 4, 5]:
    c1 = set1.filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID") \
        .unionByName(set3.filter(F.col("DECILE") >= cutoff).select("PROVIDER_ID")).distinct()
    full = c1.unionByName(ker_c2).unionByName(neph_c3).unionByName(card_c3).distinct()
    fc = full.count()
    ov = full.join(nba30_ids_c, on="PROVIDER_ID", how="inner").count()
    cap1 = full.join(nba30_ker_m1, on="PROVIDER_ID", how="inner").count()
    cap2 = full.join(m2_writers, on="PROVIDER_ID", how="inner").count()
    cap3 = full.join(m3_writers, on="PROVIDER_ID", how="inner").count()
    cap4 = full.join(m4_writers, on="PROVIDER_ID", how="inner").count()
    print(f"D{cutoff}+  {fc:>9,} {ov:>10,} {100*ov/nba30_ct:>6.1f}% | "
          f"{cap1:>8,} {100*cap1/m1_ct:>5.1f}% | {cap2:>10,} {100*cap2/m2_ct:>5.1f}% | "
          f"{cap3:>9,} {100*cap3/m3_ct:>5.1f}% | {cap4:>9,} {100*cap4/m4_ct:>5.1f}%")